In [0]:
import json, uuid, random
from datetime import datetime, timezone, timedelta

NAMESPACE = "evhns-agentops-2026"
TOPIC = "agent-traces"
conn = dbutils.secrets.get("kv-agentops", "eh-connection-string")  # never printed

kafka_opts = {
    "kafka.bootstrap.servers": f"{NAMESPACE}.servicebus.windows.net:9093",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": (
        'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="$ConnectionString" password="{conn}";'
    ),
}

AGENTS = ["sql-assistant", "support-bot", "doc-summarizer", "code-reviewer"]
MODELS = ["small-model-a", "medium-model-b", "large-model-c"]
QUERY_TYPES = ["lookup", "aggregation", "join", "summary", "chat"]
REGIONS = ["us-east", "us-central", "eu-west"]

def make_events(n):
    now = datetime.now(timezone.utc)
    events = []
    for _ in range(n):
        status = random.choices(["success", "error", "timeout"], weights=[96, 3, 1])[0]
        events.append({
            "trace_id": str(uuid.uuid4()),
            "event_time": now.isoformat(),
            "ingest_time": (now + timedelta(seconds=1)).isoformat(),
            "agent_name": random.choice(AGENTS),
            "user_id": f"user_{random.randint(1, 200):03d}",
            "model": random.choice(MODELS),
            "query_type": random.choice(QUERY_TYPES),
            "prompt_tokens": max(20, int(random.gauss(900, 300))),
            "completion_tokens": max(5, int(random.gauss(250, 100))),
            "tool_calls": random.randint(0, 4),
            "latency_ms": 30000 if status == "timeout" else max(50, int(random.gauss(1200, 400))),
            "status": status,
            "error_type": "tool_failure" if status == "error" else None,
            "region": random.choice(REGIONS),
        })
    return events

def send(n):
    rows = [(e["trace_id"], json.dumps(e)) for e in make_events(n)]
    (spark.createDataFrame(rows, ["key", "value"])
          .write.format("kafka").options(**kafka_opts)
          .option("topic", TOPIC).save())
    print(f"sent {n} events to {TOPIC}")

send(30)